# EECS 第5回 演習: Python の使い方・数値計算・データ・グラフ

電圧・電流のような数を計算し、ファイルに保存し、グラフにする。
道具は **numpy** (多くの数をまとめて計算する) と **matplotlib** (グラフを描く)。

## このノートブックの使い方

- セルは**上から順に**実行する (`Shift + Enter`)。途中から始めると「名前が見つからない」(`NameError`) と出る
- 自分で書くのは `TODO` の行だけ。`...` や `pass` の所を、正しいコードに置き換える
- `# [CHECK]` で始まるセルは確認用。**書き換えない**
  - `OK` と出れば正解
  - `NG` と出たら、その下の行に「何がおかしいか」が書いてある
- 「名前が見つからない」と出たら、`ランタイム → 前のセルを実行` を押してから、もう一度実行する
- 最初に `ファイル → ドライブにコピーを保存` を選ぶ。選ばないと、書いた内容が消える

## 全体の地図

| 課題 | 書くもの | 学ぶこと | 配点 |
|---|---|---|---|
| 課題1 | `current`, `power` | 関数を作る (`def` と `return`) | 15 |
| 課題2 | `summary` | numpy で平均・最大・最小 | 15 |
| 課題3 | `currents`, `powers` | 配列をまとめて計算する | 20 |
| 課題4 | `save_wave`, `load_wave` | CSV ファイルに保存し、読み戻す | 20 |
| 課題5 | `plot_wave` | グラフを描き、軸の名前を付ける | 20 |
| 発展1・2 | `plot_two`, `moving_average` | 凡例、移動平均。余裕があれば | 各5 |

## 進め方

- 各課題は「説明を読む → 例題セルを実行 → `TODO` を埋める → 確認セルを実行」
- 例題セルは動きを見るためのもの。書き換えずに実行すればよい
- 終わったら出力を残したまま保存し、kibaco から提出する


In [ ]:
#@title 自動セットアップ (実行するだけ。中身は隠しています)
# [SKIP] 採点の対象外
# ---- 自己確認の準備 (check / score)。1回実行すればよい ----
import pathlib
import urllib.request

if not pathlib.Path('autograde.py').exists():
    try:
        urllib.request.urlretrieve(
            'https://3tty0n.github.io/2026-scripting-lab/grading/autograde.py', 'autograde.py')
    except Exception as e:
        print('取得できませんでした:', e)

from autograde import check, score
TASK = 'e05'


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


---
## 課題1: 関数を作る (15点)

**やること**: オームの法則 (電流 = 電圧 ÷ 抵抗) を使う、2 つの関数を書く。

- `current(v, r)`: 電圧 `v` [V] と抵抗 `r` [Ω] から、電流 [A] を返す
- `power(v, r)`: 電圧 `v` と抵抗 `r` から、消費電力 [W] (電圧 × 電圧 ÷ 抵抗) を返す

**入出力の例**

| 呼び方 | 結果 |
|---|---|
| `current(5.0, 100)` | `0.05` |
| `current(12, 4)` | `3.0` |
| `power(10, 5)` | `20.0` |

**手順**

1. `current`: `v / r` を `return` する
2. `power`: `v ** 2 / r` を `return` する (`**` は累乗)

**ヒント**: 例題の `to_fahrenheit` と同じ形。`def 名前(引数):` の次の行から、空白 4 つ分下げて書く。

**つまずきやすい点**

- `print(...)` だけで `return` を書いていない → 結果が `None` になり NG になる
- `return` の行の字下げ (空白 4 つ) が無い → `IndentationError` になる

### 例題

次のセルを実行して、動きを見る。摂氏の温度を華氏に直す関数。


In [ ]:
# 例題
def to_fahrenheit(c):
    return c * 9 / 5 + 32


print(to_fahrenheit(0), to_fahrenheit(100))   # 32.0 212.0


In [ ]:
def current(v, r):
    return v / r


def power(v, r):
    return v ** 2 / r


print(current(5.0, 100), current(12, 4), power(10, 5))


In [ ]:
# [CHECK] q1
need('current', arity=2)
need('power', arity=2)

assert current(5.0, 100) is not None, 'current が値を返していない。return を書くこと'
assert abs(current(5.0, 100) - 0.05) < 1e-9, f'current(5.0, 100) が {current(5.0, 100)}。0.05 になること'
assert abs(current(12, 4) - 3.0) < 1e-9
assert power(10, 5) is not None, 'power が値を返していない。return を書くこと'
assert abs(power(10, 5) - 20.0) < 1e-9, f'power(10, 5) が {power(10, 5)}。20 になること'
assert abs(power(3, 9) - 1.0) < 1e-9
print('OK')


---
## 課題2: 平均・最大・最小 (15点)

**やること**: 測った値の並びから、平均・最大・最小の 3 つをまとめて返す関数 `summary(values)` を書く。

- 3 つの値は `(平均, 最大, 最小)` の順に、`,` でつないで返す (このまとまりをタプルと呼ぶ)

**入出力の例**

| 呼び方 | 結果 |
|---|---|
| `summary([20.5, 22.0, 19.5, 23.0])` | `(21.25, 23.0, 19.5)` |
| `summary(np.array([1, 2, 3]))` | `(2.0, 3, 1)` |

`print` すると `(np.float64(21.25), np.float64(23.0), np.float64(19.5))` のように表示されることがある。中の数が同じなら正しい。

**手順**

1. `np.mean(values)` で平均を求めて `avg` に入れる
2. `np.max(values)` と `np.min(values)` で最大と最小を求める
3. `return avg, hi, lo` のように、3 つを並べて返す

**ヒント**: numpy の関数は、リストを渡しても numpy の配列を渡しても動く。

**つまずきやすい点**

- 順番が違う (最大を先に返すなど) → 「平均・最大・最小の順」と NG になる
- `print` しただけで `return` していない → 結果が `None` になる

### 例題

次のセルを実行して、動きを見る。


In [ ]:
# 例題
temps = np.array([20.5, 22.0, 19.5, 23.0])
print(np.mean(temps), np.max(temps), np.min(temps))   # 21.25 23.0 19.5


def spread(values):
    return np.max(values) - np.min(values)


print(spread(temps))   # 3.5 (最大と最小の差)


In [ ]:
def summary(values):
    avg = np.mean(values)
    hi = np.max(values)
    lo = np.min(values)
    return avg, hi, lo


print(summary([20.5, 22.0, 19.5, 23.0]))


In [ ]:
# [CHECK] q2
need('summary', arity=1)

r = summary([20.5, 22.0, 19.5, 23.0])
assert r is not None, 'summary が値を返していない。return を書くこと'
assert len(r) == 3, '平均・最大・最小の 3 つを返すこと'
assert np.allclose(r, (21.25, 23.0, 19.5)), f'{r} になった。平均・最大・最小の順に返すこと'
assert np.allclose(summary(np.array([4, 8, 6])), (6.0, 8, 4))
assert np.allclose(summary([5.0]), (5.0, 5.0, 5.0)), '値が 1 個でも動くこと'
print('OK')


---
## 課題3: 配列をまとめて計算する (20点)

**やること**: numpy の配列を使って、たくさんの測定値を `for` なしで一度に計算する。

- `currents(voltages, r)`: 電圧の配列 `voltages` と、1 つの抵抗 `r` から、電流の配列を返す
- `powers(voltages, amps)`: 電圧の配列と電流の配列から、電力 (電圧 × 電流) の配列を返す。同じ位置どうしを掛ける

**入出力の例**

| 呼び方 | 結果 |
|---|---|
| `currents([1.0, 2.0, 3.0], 100)` | `array([0.01, 0.02, 0.03])` |
| `powers([1.0, 2.0, 3.0], [0.1, 0.2, 0.5])` | `array([0.1, 0.4, 1.5])` |

`print` すると `[0.01 0.02 0.03]` のように、`,` なしで表示される。

**手順**

1. `currents`: `np.array(voltages)` で配列にしてから `r` で割り、`return` する
2. `powers`: 両方を `np.array(...)` で配列にしてから掛け、`return` する

**ヒント**: リストのまま `[1.0, 2.0] / 100` とするとエラーになる。先に `np.array(...)` で配列にする。

**つまずきやすい点**

- `np.array(...)` を付け忘れる → `TypeError` (リストは割り算できない)
- `powers` で `voltages` と `amps` の長さが違うと計算できない。確認セルは同じ長さで試している

### 例題

次のセルを実行して、動きを見る。摂氏の配列を、まとめて華氏に直す。


In [ ]:
# 例題
c = np.array([0, 25, 100])
print(c * 9 / 5 + 32)   # [ 32.  77. 212.]


def add_offset(values, offset):
    return np.array(values) + offset


print(add_offset([1, 2, 3], 10))   # [11 12 13]


In [ ]:
def currents(voltages, r):
    return np.array(voltages) / r


def powers(voltages, amps):
    return np.array(voltages) * np.array(amps)


print(currents([1.0, 2.0, 3.0], 100))
print(powers([1.0, 2.0, 3.0], [0.1, 0.2, 0.5]))


In [ ]:
# [CHECK] q3
need('currents', arity=2)
need('powers', arity=2)

i = currents([1.0, 2.0, 3.0], 100)
assert i is not None, 'currents が値を返していない。return を書くこと'
assert isinstance(i, np.ndarray), 'currents は numpy の配列を返すこと。np.array(...) で配列にする'
assert np.allclose(i, [0.01, 0.02, 0.03]), f'{i} になった。電圧を r で割ること'
assert np.allclose(currents(np.linspace(0, 10, 6), 2), [0, 1, 2, 3, 4, 5])
p = powers([1.0, 2.0, 3.0], [0.1, 0.2, 0.5])
assert p is not None, 'powers が値を返していない。return を書くこと'
assert isinstance(p, np.ndarray), 'powers は numpy の配列を返すこと'
assert np.allclose(p, [0.1, 0.4, 1.5]), f'{p} になった。同じ位置どうしを掛けること'
print('OK')


---
## 課題4: CSV ファイルに保存し、読み戻す (20点)

**やること**: 時刻と電圧の 2 列の表を、CSV ファイル (数を `,` で区切った文字のファイル) に書く関数と、読み戻す関数を書く。

- `save_wave(path, t, v)`: 時刻の配列 `t` と電圧の配列 `v` を、ファイル `path` に 2 列の CSV で書く
- `load_wave(path)`: ファイル `path` を読み、`(t, v)` の 2 つの配列を返す

**入出力の例**

```
save_wave('wave.csv', [0.0, 0.1, 0.2], [0.0, 0.6, 0.9])
```

で、`wave.csv` の中身は次のようになる (数の表し方や桁数は違っていてよい)。

```
0.000000000000000000e+00,0.000000000000000000e+00
1.000000000000000056e-01,5.999999999999999778e-01
...
```

`load_wave('wave.csv')` → `(array([0. , 0.1, 0.2]), array([0. , 0.6, 0.9]))`

**手順**

1. `save_wave`: `np.column_stack([t, v])` で 2 列の表にし、`np.savetxt(path, 表, delimiter=',')` で書く
2. `load_wave`: `np.loadtxt(path, delimiter=',', unpack=True)` で読み、その結果を `return` する

**ヒント**: `unpack=True` を付けると、1 列目と 2 列目が別々の配列で返る。
そのまま `return` すれば `(t, v)` の形になる。

**つまずきやすい点**

- `delimiter=','` を付け忘れる → 空白区切りのファイルになり「`,` で区切ること」と NG になる
- `unpack=True` を付け忘れる → 2 列の表のまま返り、「2 つの配列を返すこと」と NG になる

### 例題

次のセルを実行して、動きを見る。1 列だけのファイルを書いて、読み戻す。


In [ ]:
# 例題
data = np.array([1.5, 2.5, 3.5])
np.savetxt('example.csv', data, delimiter=',')
print(open('example.csv').read())

back = np.loadtxt('example.csv', delimiter=',')
print(back)   # [1.5 2.5 3.5]


In [ ]:
def save_wave(path, t, v):
    table = np.column_stack([t, v])
    np.savetxt(path, table, delimiter=',')


def load_wave(path):
    return np.loadtxt(path, delimiter=',', unpack=True)


save_wave('wave.csv', [0.0, 0.1, 0.2], [0.0, 0.6, 0.9])
print(open('wave.csv').read())
print(load_wave('wave.csv'))


In [ ]:
# [CHECK] q4
need('save_wave', arity=3)
need('load_wave', arity=1)
import os as _os, tempfile as _tempfile

_d = _tempfile.mkdtemp()
_p = _os.path.join(_d, 'w.csv')
save_wave(_p, [0.0, 0.5, 1.0, 1.5], [1.0, -1.0, 2.0, 0.25])
assert _os.path.exists(_p), 'save_wave がファイルを作っていない。np.savetxt で書くこと'
_lines = open(_p).read().split()
assert len(_lines) == 4, f'{len(_lines)} 行になった。1 行に時刻と電圧を 1 組ずつ書くこと'
assert all(',' in s for s in _lines), '列は , で区切ること (delimiter=",")'
r = load_wave(_p)
assert r is not None, 'load_wave が値を返していない。return を書くこと'
assert len(r) == 2, '時刻と電圧の 2 つの配列を返すこと (unpack=True)'
assert np.allclose(r[0], [0.0, 0.5, 1.0, 1.5]) and np.allclose(r[1], [1.0, -1.0, 2.0, 0.25]), \
    '読み戻した値が、書いた値と違う'
with open(_p, 'w') as _f:
    _f.write('0,3\n1,4\n2,5\n')
assert np.allclose(load_wave(_p)[1], [3, 4, 5]), '手で書いた CSV も読めること'
print('OK')


---
## 課題5: グラフを描く (20点)

**やること**: 時刻 `t` と電圧 `v` の折れ線グラフを描く関数 `plot_wave(t, v)` を書く。

- 横軸の名前は `time [s]`、縦軸の名前は `voltage [V]` にし、題 (タイトル) も付ける
- 描いた図 `fig` と描く場所 `ax` の 2 つを返す (確認セルが、線や軸の名前を調べるため)

**入出力の例**

`fig, ax = plot_wave(t, v)` を実行すると、横軸が時刻、縦軸が電圧の、線が 1 本のグラフが表示される。

**手順**

1. `fig, ax = plt.subplots()` で図と描く場所を作る
2. `ax.plot(t, v)` で線を描く
3. `ax.set_xlabel(...)`、`ax.set_ylabel(...)`、`ax.set_title(...)` で名前を付ける
4. `return fig, ax` で 2 つを返す

**ヒント**: 例題の `plot_cos` と同じ形。違うのは、描く値と軸の名前だけ。

**つまずきやすい点**

- 軸の名前を日本語にすると、Colab では □ に化ける。英語で書く
- `return fig, ax` を忘れる → 「fig と ax の 2 つを返すこと」と NG になる
- `ax.plot` を 2 回書くと線が 2 本になり、「線は 1 本」と NG になる

### 例題

次のセルを実行して、動きを見る。余弦波 (cos) のグラフ。


In [ ]:
# 例題
def plot_cos(t):
    fig, ax = plt.subplots()
    ax.plot(t, np.cos(2 * np.pi * t))
    ax.set_xlabel('time [s]')
    ax.set_title('cosine')
    return fig, ax


fig, ax = plot_cos(np.linspace(0, 2, 200))
plt.show()


In [ ]:
t = np.linspace(0, 1, 200)
v = 2.0 * np.sin(2 * np.pi * 3 * t)   # 振幅 2 V、3 Hz の正弦波


def plot_wave(t, v):
    fig, ax = plt.subplots()
    ax.plot(t, v)
    ax.set_xlabel('time [s]')
    ax.set_ylabel('voltage [V]')
    ax.set_title('3 Hz sine wave')
    return fig, ax


fig, ax = plot_wave(t, v)
plt.show()


In [ ]:
# [CHECK] q5
need('plot_wave', arity=2)

_t = np.linspace(0, 1, 50)
_v = np.cos(2 * np.pi * _t)
r = plot_wave(_t, _v)
assert r is not None and len(r) == 2, 'fig と ax の 2 つを返すこと (return fig, ax)'
_fig, _ax = r
assert len(_ax.lines) == 1, f'線が {len(_ax.lines)} 本ある。線は 1 本 (ax.plot を 1 回)'
assert np.allclose(_ax.lines[0].get_xdata(), _t), '横軸に t を渡すこと (ax.plot(t, v))'
assert np.allclose(_ax.lines[0].get_ydata(), _v), '縦軸に v を渡すこと (ax.plot(t, v))'
assert _ax.get_xlabel(), '横軸の名前が無い。ax.set_xlabel(...) を書くこと'
assert _ax.get_ylabel(), '縦軸の名前が無い。ax.set_ylabel(...) を書くこと'
assert _ax.get_title(), '題が無い。ax.set_title(...) を書くこと'
plt.close('all')
print('OK')


---
## 発展1: 線を重ねて凡例を付ける (5点)

**やること**: 2 本の線を 1 つのグラフに重ね、それぞれの名前 (凡例) を出す関数
`plot_two(t, v1, v2, label1, label2)` を書く。`fig, ax` を返す。

**入出力の例**

`plot_two(t, np.sin(t), np.cos(t), 'sin', 'cos')` → 線が 2 本、右上などに `sin` と `cos` の凡例が出る

**手順**

1. `fig, ax = plt.subplots()`
2. `ax.plot(t, v1, label=label1)` と `ax.plot(t, v2, label=label2)`
3. `ax.legend()` で凡例を出す
4. `return fig, ax`

**つまずきやすい点**

- `label=` を付けずに `ax.legend()` を呼ぶ → 凡例が空になり NG になる

### 例題

線 1 本に名前を付けて、凡例を出す例。


In [ ]:
# 例題
fig, ax = plt.subplots()
ax.plot(t, np.sin(2 * np.pi * t), label='1 Hz')
ax.legend()
plt.show()


In [ ]:
def plot_two(t, v1, v2, label1, label2):
    fig, ax = plt.subplots()
    ax.plot(t, v1, label=label1)
    ax.plot(t, v2, label=label2)
    ax.legend()
    return fig, ax


fig, ax = plot_two(t, np.sin(2 * np.pi * t), np.sin(2 * np.pi * 2 * t), '1 Hz', '2 Hz')
plt.show()


In [ ]:
# [CHECK] b1
need('plot_two', arity=5)

_t = np.linspace(0, 1, 30)
_fig, _ax = plot_two(_t, _t, 2 * _t, 'a', 'b')
assert len(_ax.lines) == 2, f'線が {len(_ax.lines)} 本。2 本描くこと'
assert np.allclose(_ax.lines[1].get_ydata(), 2 * _t), '2 本目は v2 を描くこと'
_leg = _ax.get_legend()
assert _leg is not None, '凡例が無い。ax.legend() を呼ぶこと'
assert [x.get_text() for x in _leg.get_texts()] == ['a', 'b'], 'label=label1, label=label2 を付けること'
plt.close('all')
print('OK')


---
## 発展2: 移動平均でノイズをならす (5点)

**やること**: 測定値のがたつき (ノイズ) をならす関数 `moving_average(v, n)` を書く。
となり合う `n` 個の平均を、1 つずつずらしながら求めた配列を返す (これを移動平均という)。

**入出力の例**

| 呼び方 | 結果 |
|---|---|
| `moving_average([1, 2, 3, 4, 5], 2)` | `array([1.5, 2.5, 3.5, 4.5])` |
| `moving_average([1, 2, 3, 4, 5], 3)` | `array([2., 3., 4.])` |

結果の長さは `len(v) - n + 1` になる。

**手順**

1. `np.ones(n) / n` で、`1/n` が `n` 個並んだ配列を作る
2. `np.convolve(v, それ, mode='valid')` を `return` する

**ヒント**: `np.convolve` は「重みを掛けて足す」をずらしながら繰り返す関数。
重みが全部 `1/n` なら、`n` 個の平均になる。

**つまずきやすい点**

- `mode='valid'` を付け忘れる → 端の分だけ長くなり「長さが違う」と NG になる

### 例題

ノイズの乗った正弦波を作る。下の課題で、これをならしてグラフで比べる。


In [ ]:
# 例題
rng = np.random.default_rng(0)
noisy = np.sin(2 * np.pi * t) + 0.3 * rng.standard_normal(len(t))
print(noisy[:5].round(2))


In [ ]:
def moving_average(v, n):
    return np.convolve(v, np.ones(n) / n, mode='valid')


print(moving_average([1, 2, 3, 4, 5], 2))
smooth = moving_average(noisy, 15)
fig, ax = plt.subplots()
ax.plot(t, noisy, label='noisy')
ax.plot(t[:len(smooth)], smooth, label='moving average')
ax.legend()
plt.show()


In [ ]:
# [CHECK] b2
need('moving_average', arity=2)

r = moving_average([1, 2, 3, 4, 5], 2)
assert r is not None, 'moving_average が値を返していない。return を書くこと'
assert len(r) == 4, f'長さが {len(r)}。len(v) - n + 1 になること (mode="valid")'
assert np.allclose(r, [1.5, 2.5, 3.5, 4.5])
assert np.allclose(moving_average([1, 2, 3, 4, 5], 3), [2, 3, 4])
assert np.allclose(moving_average(np.full(10, 7.0), 4), 7.0), '同じ値の平均は同じ値'
print('OK')


---
## 自己確認

`score()` で全部まとめて確かめる。何度やってもよい。
`NG` の行に、次に何をすればよいかが出る。


In [ ]:
# [SKIP] 採点の対象外
score()


---
## 提出

提出は kibaco から。`score()` と採点ページは下書き確認用。

1. `ランタイム → すべてのセルを実行` で上から実行し、出力を残す
2. `ファイル → .ipynb をダウンロード`
3. 締切までに kibaco へ出す

確認セルが全部 `OK` なら提出してよい。確認セルを消しても、教員は別の検査で採点する。
